# Brain Tumor MRI Classification - Exploratory Data Analysis

## Dataset Overview
This notebook performs comprehensive EDA on the Brain Tumor MRI dataset containing 4 classes:
- **Glioma** - Tumors from glial cells
- **Meningioma** - Tumors from meninges
- **Pituitary** - Pituitary gland tumors  
- **Healthy** - Normal brain scans

**Objectives:**
1. Analyze class distribution and dataset balance
2. Explore image properties (dimensions, channels, file sizes)
3. Visualize sample images from each class
4. Identify preprocessing requirements
5. Generate insights for model development

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. Dataset Loading and Structure

In [ ]:
dataset_path = Path('Dataset/brain_tumor_dataset')

classes = ['glioma', 'healthy', 'meningioma', 'pituitary']

print(f"Dataset Path: {dataset_path.absolute()}")
print(f"Classes: {classes}")
print(f"\nDataset Structure:")
for class_name in classes:
    class_path = dataset_path / class_name
    if class_path.exists():
        print(f"  ✓ {class_name}/")
    else:
        print(f"  ✗ {class_name}/ - NOT FOUND")

## 2. Class Distribution Analysis

In [ ]:
class_counts = {}
all_image_paths = []

for class_name in classes:
    class_path = dataset_path / class_name
    image_files = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png')) + list(class_path.glob('*.jpeg'))
    class_counts[class_name] = len(image_files)
    all_image_paths.extend([(str(img), class_name) for img in image_files])

df_images = pd.DataFrame(all_image_paths, columns=['path', 'class'])

print("=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)
for class_name, count in class_counts.items():
    percentage = (count / sum(class_counts.values())) * 100
    print(f"{class_name.capitalize():15s}: {count:5d} images ({percentage:5.2f}%)")
print("=" * 60)
print(f"{'Total':15s}: {sum(class_counts.values()):5d} images")
print("=" * 60)

max_count = max(class_counts.values())
min_count = min(class_counts.values())
imbalance_ratio = max_count / min_count
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")
if imbalance_ratio < 1.5:
    print("✓ Dataset is well-balanced")
elif imbalance_ratio < 3:
    print("⚠ Slight class imbalance - consider data augmentation")
else:
    print("⚠ Significant class imbalance - use weighted loss or resampling")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

axes[0].bar(class_counts.keys(), class_counts.values(), color=colors, alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Images', fontsize=12, fontweight='bold')
axes[0].set_title('Class Distribution - Bar Chart', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, (class_name, count) in enumerate(class_counts.items()):
    axes[0].text(i, count + 30, str(count), ha='center', fontweight='bold')

axes[1].pie(class_counts.values(), labels=class_counts.keys(), autopct='%1.1f%%', 
            colors=colors, startangle=90, explode=[0.05]*4, shadow=True)
axes[1].set_title('Class Distribution - Pie Chart', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Image Properties Analysis

In [ ]:
print("Analyzing image properties (sampling 100 images per class)...\n")

image_properties = []

for class_name in classes:
    class_path = dataset_path / class_name
    image_files = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png')) + list(class_path.glob('*.jpeg'))
    
    sample_size = min(100, len(image_files))
    sampled_images = np.random.choice(image_files, sample_size, replace=False)
    
    for img_path in sampled_images:
        try:
            img = Image.open(img_path)
            width, height = img.size
            mode = img.mode
            file_size = os.path.getsize(img_path) / 1024
            
            image_properties.append({
                'class': class_name,
                'width': width,
                'height': height,
                'aspect_ratio': width / height,
                'mode': mode,
                'file_size_kb': file_size
            })
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

df_properties = pd.DataFrame(image_properties)

print("=" * 80)
print("IMAGE PROPERTIES SUMMARY")
print("=" * 80)
print(f"\nSample Size: {len(df_properties)} images analyzed\n")
print(df_properties.describe())

In [ ]:
print("\n" + "=" * 80)
print("IMAGE DIMENSIONS BY CLASS")
print("=" * 80)
for class_name in classes:
    class_data = df_properties[df_properties['class'] == class_name]
    print(f"\n{class_name.upper()}:")
    print(f"  Width  - Min: {class_data['width'].min():.0f}, Max: {class_data['width'].max():.0f}, Mean: {class_data['width'].mean():.0f}")
    print(f"  Height - Min: {class_data['height'].min():.0f}, Max: {class_data['height'].max():.0f}, Mean: {class_data['height'].mean():.0f}")
    print(f"  Aspect Ratio: {class_data['aspect_ratio'].mean():.3f}")
    print(f"  File Size (KB): {class_data['file_size_kb'].mean():.2f}")

In [ ]:
print("\n" + "=" * 80)
print("IMAGE MODES (Color Channels)")
print("=" * 80)
mode_counts = df_properties['mode'].value_counts()
for mode, count in mode_counts.items():
    percentage = (count / len(df_properties)) * 100
    print(f"{mode:10s}: {count:4d} images ({percentage:5.2f}%)")
    if mode == 'RGB':
        print("           (3 channels - color images)")
    elif mode == 'L':
        print("           (1 channel - grayscale images)")
    elif mode == 'RGBA':
        print("           (4 channels - color + alpha)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

for idx, class_name in enumerate(classes):
    ax = axes[idx // 2, idx % 2]
    class_data = df_properties[df_properties['class'] == class_name]
    
    ax.scatter(class_data['width'], class_data['height'], alpha=0.6, s=50, color=colors[idx])
    ax.set_xlabel('Width (pixels)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Height (pixels)', fontsize=11, fontweight='bold')
    ax.set_title(f'{class_name.capitalize()} - Image Dimensions', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    
    mean_width = class_data['width'].mean()
    mean_height = class_data['height'].mean()
    ax.axvline(mean_width, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Mean W: {mean_width:.0f}')
    ax.axhline(mean_height, color='blue', linestyle='--', linewidth=2, alpha=0.7, label=f'Mean H: {mean_height:.0f}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df_properties.boxplot(column='width', by='class', ax=axes[0], patch_artist=True)
axes[0].set_title('Width Distribution by Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Width (pixels)', fontsize=12, fontweight='bold')
plt.sca(axes[0])
plt.xticks(rotation=0)

df_properties.boxplot(column='height', by='class', ax=axes[1], patch_artist=True)
axes[1].set_title('Height Distribution by Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Height (pixels)', fontsize=12, fontweight='bold')
plt.sca(axes[1])
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 4. Sample Image Visualization

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
fig.suptitle('Sample Images from Each Class (5 samples per class)', fontsize=16, fontweight='bold', y=0.995)

for idx, class_name in enumerate(classes):
    class_path = dataset_path / class_name
    image_files = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png')) + list(class_path.glob('*.jpeg'))
    
    sample_images = np.random.choice(image_files, 5, replace=False)
    
    for col, img_path in enumerate(sample_images):
        try:
            img = Image.open(img_path)
            axes[idx, col].imshow(img, cmap='gray' if img.mode == 'L' else None)
            axes[idx, col].axis('off')
            
            if col == 0:
                axes[idx, col].set_ylabel(class_name.capitalize(), fontsize=13, fontweight='bold', rotation=90, labelpad=10)
            
            width, height = img.size
            axes[idx, col].set_title(f'{width}x{height}', fontsize=9)
        except Exception as e:
            axes[idx, col].text(0.5, 0.5, 'Error', ha='center', va='center')
            axes[idx, col].axis('off')

plt.tight_layout()
plt.show()

## 5. Pixel Intensity Analysis

In [ ]:
print("Analyzing pixel intensities (sampling 50 images per class)...\n")

intensity_stats = []

for class_name in classes:
    class_path = dataset_path / class_name
    image_files = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png')) + list(class_path.glob('*.jpeg'))
    
    sample_size = min(50, len(image_files))
    sampled_images = np.random.choice(image_files, sample_size, replace=False)
    
    for img_path in sampled_images:
        try:
            img = Image.open(img_path).convert('L')
            img_array = np.array(img)
            
            intensity_stats.append({
                'class': class_name,
                'mean_intensity': img_array.mean(),
                'std_intensity': img_array.std(),
                'min_intensity': img_array.min(),
                'max_intensity': img_array.max()
            })
        except Exception as e:
            continue

df_intensity = pd.DataFrame(intensity_stats)

print("=" * 80)
print("PIXEL INTENSITY STATISTICS BY CLASS")
print("=" * 80)
for class_name in classes:
    class_data = df_intensity[df_intensity['class'] == class_name]
    print(f"\n{class_name.upper()}:")
    print(f"  Mean Intensity: {class_data['mean_intensity'].mean():.2f} ± {class_data['mean_intensity'].std():.2f}")
    print(f"  Std Intensity:  {class_data['std_intensity'].mean():.2f}")
    print(f"  Range: [{class_data['min_intensity'].mean():.0f}, {class_data['max_intensity'].mean():.0f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for class_name in classes:
    class_data = df_intensity[df_intensity['class'] == class_name]
    axes[0].hist(class_data['mean_intensity'], bins=30, alpha=0.6, label=class_name.capitalize())

axes[0].set_xlabel('Mean Pixel Intensity', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0].set_title('Mean Pixel Intensity Distribution by Class', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

df_intensity.boxplot(column='mean_intensity', by='class', ax=axes[1], patch_artist=True)
axes[1].set_title('Mean Intensity by Class - Box Plot', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Mean Pixel Intensity', fontsize=12, fontweight='bold')
plt.sca(axes[1])
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 6. Key Findings and Recommendations

In [ ]:
print("="*80)
print("KEY FINDINGS & RECOMMENDATIONS")
print("="*80)

print("\n📊 DATASET CHARACTERISTICS:")
print(f"   • Total images: {sum(class_counts.values())}")
print(f"   • Classes: {len(classes)}")
print(f"   • Class balance ratio: {imbalance_ratio:.2f}:1")

print("\n🖼️  IMAGE PROPERTIES:")
print(f"   • Average dimensions: {df_properties['width'].mean():.0f}x{df_properties['height'].mean():.0f} pixels")
print(f"   • Dimension range: [{df_properties['width'].min():.0f}-{df_properties['width'].max():.0f}] x [{df_properties['height'].min():.0f}-{df_properties['height'].max():.0f}]")
print(f"   • Primary color mode: {df_properties['mode'].mode()[0]}")
print(f"   • Average file size: {df_properties['file_size_kb'].mean():.2f} KB")

print("\n💡 PREPROCESSING RECOMMENDATIONS:")
print("   1. Image Resizing:")
recommended_size = 224
print(f"      → Resize all images to {recommended_size}x{recommended_size} (standard for transfer learning)")
print("      → Alternative: 128x128 for faster training, 299x299 for inception models")

print("\n   2. Normalization:")
print("      → Normalize pixel values to [0, 1] range: divide by 255")
print("      → Or standardize: (pixel - mean) / std")
print("      → For transfer learning: use ImageNet mean/std")

print("\n   3. Data Augmentation (to improve generalization):")
print("      → Rotation: ±15-20 degrees")
print("      → Horizontal flip: Yes")
print("      → Zoom: 10-15%")
print("      → Brightness/Contrast: slight variations")
print("      ⚠️  Avoid vertical flips (not anatomically meaningful)")

print("\n   4. Train/Validation/Test Split:")
print("      → Recommended: 70% train, 15% validation, 15% test")
print("      → Alternative: 80% train, 10% validation, 10% test")
print("      → Use stratified split to maintain class balance")

print("\n   5. Color Handling:")
if 'RGB' in df_properties['mode'].values:
    print("      → Convert all images to RGB for consistency")
    print("      → Or convert to grayscale if color isn't critical")

print("\n🎯 MODEL SELECTION SUGGESTIONS:")
print("   • Transfer Learning (Recommended):")
print("      - VGG16/VGG19 - Good baseline, well-tested")
print("      - ResNet50/ResNet101 - Better for deeper learning")
print("      - EfficientNet - Best accuracy/efficiency trade-off")
print("      - DenseNet - Good for medical imaging")
print("\n   • Custom CNN (Alternative):")
print("      - 3-5 convolutional blocks")
print("      - Batch normalization + Dropout for regularization")
print("      - Suitable for learning specific features")

print("\n📈 EVALUATION METRICS:")
print("   • Accuracy (overall performance)")
print("   • Precision, Recall, F1-score (per class)")
print("   • Confusion Matrix (misclassification patterns)")
print("   • ROC-AUC curves (class discrimination ability)")
print("   ⚠️  Focus on recall for tumor classes (minimize false negatives)")

print("\n" + "="*80)
print("✅ EDA COMPLETE - Ready for model development!")
print("="*80)

## Summary

This EDA provides comprehensive insights into the brain tumor MRI dataset:

**Dataset Quality:** Well-balanced multi-class dataset suitable for CNN training

**Next Steps:**
1. Implement data preprocessing pipeline
2. Create train/val/test splits
3. Build and train CNN models
4. Evaluate model performance
5. Optimize and fine-tune

**Note:** This dataset is for research and educational purposes only. Clinical applications require medical validation.